# Experimental: the LR-QAOA benchmark in the X frame, on IBM

Does the frame of the benchmark explain the $X$ memory results? The benchmark measures a code's checks as
$Z\cdots Z$ terms: the data start in $|+\rangle$, and the late layers - where the LR ramp is close to a
computational-basis state - are hurt most by errors that **flip** the data, the ones $Z$ memory fails on. The
**X frame** is the same algorithm conjugated by $H$ on every data qubit ($X\cdots X$ terms, `rz` mixer, data from
$|0\rangle$, read in $X$), built natively so the data really sit in the $X$ frame through every ancilla readout:
there a **phase** error does what a flip did, which is what $X$ memory fails on. Noiselessly both give the same
$r$, so $r_{\rm ovl}$ of the two frames compares directly.

**The prediction:** at each $p = R$, the X frame ranks the patches like the $X$ memory experiment and the Z frame
like the $Z$ one: $\rho(r^X_{\rm ovl}, P_L^X) > \rho(r^Z_{\rm ovl}, P_L^X)$, and the reverse for $Z$. If both frames
rank the patches alike, the frame is not what separates the two memories.

Both frames run **interleaved in the same jobs**, on the patches of the 2026-09-14 scan. Run
`benchmark_qec_memory.ipynb` on the same patches in the same session, so section 5 has its memory data.

**Removable:** `qecbench.xframe`, `tests/test_xframe.py`, this notebook (and its function in `make_notebooks.py`),
and the folders `data/manifests/xframe/` and `data/results/<backend>/surface_code/xframe/`. Nothing else uses them.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))          # not needed after `pip install -e .`
DATA, FIGURES = ROOT / "data", ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr

from qecbench import codes
from qecbench import xframe as xf
from qecbench.backends import IBMBackend
from qecbench.layout import surface_code_placements

## 1. Configuration

`SUBMIT = False` builds, transpiles and prices everything against the device (an account is needed to read its
target, nothing is sent); `True` sends it.

In [ ]:
BACKEND_NAME = "ibm_phoenix"
ACCOUNT      = "mcm-primitives"
SCAN_ANCHORS = {3: [(7, 2), (7, 3), (6, 3), (0, 3), (0, 4), (7, 5), (6, 4), (2, 5), (3, 7), (7, 7), (3, 2)],
                5: [(0, 5), (0, 4), (3, 5), (1, 5), (1, 4), (2, 4), (3, 4), (2, 5)]}
DISTANCES    = [3, 5]
DEPTHS       = [1, 2, 3, 5, 7, 10]     # the memory experiment's round counts, so every p has its R = p
SHOTS        = 1000
DELTA        = 0.5
SUBMIT       = False                   # True sends the jobs and consumes QPU time

## 2. Patches and circuits

The same placements as the benchmark and the memory experiment (`layout.surface_code_placements` from the scan's
anchors). Below, what the transpiler makes of one circuit in each frame: the X frame has no `sx` in the mixer
(`rz` is virtual) but a Hadamard on each data qubit around every block of CZs, and on this device the two come out
with the same number of `sx` pulses - so the frames differ in where the data sit, not in how many gates they get.

In [ ]:
backend = IBMBackend(BACKEND_NAME, account=ACCOUNT, optimization_level=1, dynamical_decoupling=False)
G = backend.coupling_graph()
patches = [p for d in DISTANCES for p in surface_code_placements(G, codes.surface_code(d), anchors=SCAN_ANCHORS[d])]
print(f"{len(patches)} patches: " + ", ".join(f"{p.code.name}@{min(p.data_qubits)}" for p in patches))

xplan = xf.plan(backend, patches, DEPTHS, shots=SHOTS, delta=DELTA)
est = xplan["estimate"]
n = len(xplan["circuits"])
print(f"\n{n} circuits = {len(patches)} patches x {len(DEPTHS)} depths x 2 frames, "
      f"{-(-n // backend.max_circuits_per_job)} jobs of up to {backend.max_circuits_per_job}")
print(f"estimate: {est['total']:.1f} QPU s = ${est['usd']:.2f}   ({est['notes'][0]})")

i = next(k for k, t in enumerate(xplan["tasks"]) if t["depth"] == 3)
print("\ntranspiled gates, first patch at p = 3:")
for frame, qc in (("Z", xplan["circuits"][i]), ("X", xplan["circuits"][i + 1])):
    print(f"  {frame} frame: " + ", ".join(f"{k} {v}" for k, v in sorted(qc.count_ops().items())))

## 3. Submit

A calibration snapshot is taken first, as for every run. The manifest (`data/manifests/xframe/`) is written after
every job. Then run the memory experiment on the same patches.

In [ ]:
if SUBMIT:
    backend.calibration(save_dir=DATA / "calibration")
    manifest_path = xf.submit(xplan, backend, manifest_dir=DATA / "manifests" / "xframe")
else:
    print("SUBMIT = False - nothing sent")

## 4. Harvest

The newest X-frame manifest; harvesting again later adds the jobs that were still running.

In [ ]:
manifests = sorted((DATA / "manifests" / "xframe").glob("*_xframe.json"))
if manifests:
    saved = xf.harvest(manifests[-1], backend, data_dir=DATA / "results")
else:
    print("no X-frame manifest yet")

## 5. Both frames against the memory experiment of the same day

For each $p = R$ and each basis, the Spearman $\rho$ between $-r_{\rm ovl}(p)$ of each frame and the measured
$P_L(R)$ over the $d = 3$ patches (positive = the patch the frame rates higher keeps its logical qubit better).
The prediction holds if the diagonal of each table (Z frame with $Z$ memory, X frame with $X$ memory) beats the
off-diagonal. Below it: do the two frames even rate the patches differently?

In [ ]:
runs = sorted((DATA / "results" / BACKEND_NAME / "surface_code" / "xframe").glob("*_xframe.json"))
assert runs, "no harvested X-frame run yet"
frames, run = xf.load(runs[-1])
day = run["created"][:10].replace("-", "")
memory_files = sorted((DATA / "results" / BACKEND_NAME / "surface_code" / "memory").glob(f"{day}_*_memory.json"))
assert memory_files, f"no memory run on {run['created'][:10]} - run benchmark_qec_memory.ipynb on the same patches"
P_L = {}
for rec in json.loads(memory_files[-1].read_text())["results"]:
    key = tuple(rec["parameters"]["data_qubits"])
    P_L.setdefault(rec["parameters"]["basis"], {}).setdefault(key, {})[rec["parameters"]["rounds"]] = \
        rec["benchmark"]["logical_error_rate"]
print(f"X-frame run {runs[-1].name}  +  memory run {memory_files[-1].name}")

d3 = [k for k in frames["Z"] if len(k) == 9 and k in P_L["Z"]]
depths = sorted(set(DEPTHS) & set(next(iter(P_L["Z"].values()))))
rho = {}
for basis in ("Z", "X"):
    print(f"\nrho(-r_ovl(p), P_L^{basis}(R = p)), {len(d3)} surface_d3 patches")
    print(f"{'frame':>8} " + "".join(f"{'p=R=' + str(p):>10}" for p in depths))
    for frame in ("Z", "X"):
        row = [spearmanr([-frames[frame][k][p] for k in d3], [P_L[basis][k][p] for k in d3])[0] for p in depths]
        rho[(frame, basis)] = row
        mark = "  <- same frame as the memory" if frame == basis else ""
        print(f"{frame:>8} " + "".join(f"{v:>+10.2f}" for v in row) + mark)

print("\ndo the frames rate the patches differently? rho(r_ovl Z frame, r_ovl X frame) and the mean X - Z")
for p in depths:
    a, b = [frames["Z"][k][p] for k in d3], [frames["X"][k][p] for k in d3]
    print(f"  p = {p:>2}: rho = {spearmanr(a, b)[0]:+.2f}   mean r_ovl X - Z = {np.mean(b) - np.mean(a):+.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, basis in zip(axes, ("Z", "X")):
    for frame, style in (("Z", "o-"), ("X", "s--")):
        ax.plot(depths, rho[(frame, basis)], style, lw=2.5 if frame == basis else 1.5, label=f"{frame} frame")
    ax.axhline(0, color="k", lw=0.8)
    ax.set(title=f"agreement with the {basis} memory", xlabel="p = R", ylim=(-1, 1))
    ax.grid(alpha=0.3)
    ax.legend()
axes[0].set_ylabel(r"Spearman $\rho$")
fig.tight_layout()
fig.savefig(FIGURES / f"{BACKEND_NAME}_xframe_vs_memory.pdf", bbox_inches="tight")
plt.show()

### $r_{\rm ovl}$ against the logical error rate, patch by patch

Every $d = 3$ patch at every $p = R$: its $r_{\rm ovl}$ after $p$ layers in each frame against its measured logical
error rate after $R = p$ rounds (top: $Z$ memory, bottom: $X$ memory). A thin line joins the two frames of one patch;
the frame matching the memory's basis is outlined. A good predictor puts the points on a falling line.

In [ ]:
styles = {"Z": dict(marker="o", color="#2f6f9f"), "X": dict(marker="s", color="#b8560f")}
fig, axes = plt.subplots(2, len(depths), figsize=(3.2 * len(depths), 6.8), sharey="row", squeeze=False)
for i, basis in enumerate(("Z", "X")):
    for j, p in enumerate(depths):
        ax = axes[i, j]
        for k in d3:
            ax.plot([frames["Z"][k][p], frames["X"][k][p]], [P_L[basis][k][p]] * 2, color="0.8", lw=0.8, zorder=1)
        for frame in ("Z", "X"):
            ro = [frames[frame][k][p] for k in d3]
            pl = [P_L[basis][k][p] for k in d3]
            r = spearmanr([-v for v in ro], pl)[0]
            ax.scatter(ro, pl, s=38, alpha=0.9, zorder=2, label=rf"{frame} frame, $\rho$ = {r:+.2f}", **styles[frame],
                       edgecolor="k" if frame == basis else "none", linewidth=0.8)
        ax.set_yscale("log")
        ax.grid(alpha=0.3)
        ax.set_title(f"p = R = {p}", fontsize=10)
        ax.legend(fontsize=7, loc="best")
        if i == 1:
            ax.set_xlabel(r"$r_{\rm ovl}(p)$")
    axes[i, 0].set_ylabel(rf"$P_L^{basis}(R)$")
fig.tight_layout()
fig.savefig(FIGURES / f"{BACKEND_NAME}_xframe_rovl_vs_logical_error.pdf", bbox_inches="tight")
plt.show()

### The best patches, depth by depth

The `N_BEST` patches that kept the $X$ logical qubit best (lowest $P_L^X$, ranked at every round count and
averaged), each followed from $p = R = 1$ to $10$: its $r_{\rm ovl}(p)$ against $P_L^X(R = p)$, the X frame solid and
labelled with the depth, the Z frame of the same patch dashed. `N_WORST` adds the patches at the other end, thinner
and with triangles (`0` leaves them out): if the X frame measures what the memory does, they continue the best
patches' curve further down rather than lying on a curve of their own.

In [ ]:
N_BEST, N_WORST = 3, 3                     # N_WORST = 0: the best patches only
mean_rank = {k: np.mean([sorted(d3, key=lambda q: P_L["X"][q][p]).index(k) + 1 for p in depths]) for k in d3}
order = sorted(d3, key=lambda k: mean_rank[k])
best, worst = order[:N_BEST], (order[-N_WORST:] if N_WORST else [])
print("best by X memory:  " + ", ".join(f"@{min(k)} (mean rank {mean_rank[k]:.1f})" for k in best))
if worst:
    print("worst by X memory: " + ", ".join(f"@{min(k)} (mean rank {mean_rank[k]:.1f})" for k in worst))

fig, ax = plt.subplots(figsize=(8.5, 5.8))
groups = [(best, plt.get_cmap("tab10").colors, dict(marker="s", lw=2, ms=7), dict(lw=1, ms=5, alpha=0.55)),
          (worst, plt.get_cmap("Dark2").colors[3:], dict(marker="^", lw=1.1, ms=6, alpha=0.8),
           dict(lw=0.8, ms=4, alpha=0.4))]
for patches_, colours, x_style, z_style in groups:
    for colour, k in zip(colours, patches_):
        pl = [P_L["X"][k][p] for p in depths]
        ax.plot([frames["X"][k][p] for p in depths], pl, "-", color=colour, label=f"@{min(k)}, X frame", **x_style)
        ax.plot([frames["Z"][k][p] for p in depths], pl, "o--", color=colour, label=f"@{min(k)}, Z frame", **z_style)
        for p, xv, yv in zip(depths, [frames["X"][k][p] for p in depths], pl):
            ax.annotate(f"{p}", (xv, yv), textcoords="offset points", xytext=(5, 4), fontsize=8, color=colour)
ax.set(xlabel=r"$r_{\rm ovl}(p)$", ylabel=r"$P_L^X(R = p)$", yscale="log",
       title=f"the {N_BEST} best" + (f" and {N_WORST} worst" if worst else "")
             + f" patches by X memory, p = R = {depths[0]} ... {depths[-1]}")
ax.grid(alpha=0.3)
ax.legend(fontsize=7.5, ncol=2)
fig.tight_layout()
fig.savefig(FIGURES / f"{BACKEND_NAME}_xframe_best_patches.pdf", bbox_inches="tight")
plt.show()